In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["ANTHROPIC_API_KEY"]=os.getenv("ANTHROPIC_API_KEY")
os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")

### StateBackend


In [13]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [33]:
from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3:8b",
    temperature=0,
)

In [55]:
from deepagents import backends
# ==================================
# Creating agent
# ==================================



# Without backend
agent1 = create_deep_agent(model='google_genai:gemini-3.8-flash')

# With backend
agent = create_deep_agent(model=model, backend=StateBackend())

In [56]:
# ====================================
# Invoke agent and ask for write file
# ====================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /note/todo1.txt with exactly this content:\n"
                    "1. Recored Video\n"
                    "2. Edit Vedio\n"
                    "3. Upload video\n"
                    "Then tell me you've done it."
                ),
            }
        ]
    }
)

In [57]:
print("=========================Result=========================")
print(result)
print("Content:------")
print(result["messages"][-1].content)

print(result.get("files", {}))

=========================Result=========================
{'messages': [HumanMessage(content="Create a file at /note/todo1.txt with exactly this content:\n1. Recored Video\n2. Edit Vedio\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='2a01168e-1cca-4af0-9407-743fc8883984'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-02T08:48:52.256511Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9483720708, 'load_duration': 5647166, 'prompt_eval_count': 2300, 'prompt_eval_duration': 377639000, 'eval_count': 186, 'eval_duration': 9044503000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0fbcd-2d11-7d61-9931-6267d8b34583-0', tool_calls=[{'name': 'write_file', 'args': {'content': '1. Recored Video\n2. Edit Vedio\n3. Upload video', 'file_path': '/note/todo1.txt'}, 'id': '6ab37312-ed80-4d8f-9b97-f23c25af7b15', 'type': 'tool_call'}], invalid_to

In [ ]:
read_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "read the file context /note/todo1.txt and show here"
                ),
            }
        ],
        "files": result.get("files", {}),   # <-- pass the virtual filesystem along
    }
)


print("======================READ===Result=========================")
print(read_result)
print("Content:------")
print(read_result["messages"][-1])

======================READ===Result=========================
{'messages': [HumanMessage(content='read the file context /note/todo1.txt and show here', additional_kwargs={}, response_metadata={}, id='508d02b0-d2bf-4965-a241-b34f5ab081d4'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-02T08:50:35.152339Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17531887750, 'load_duration': 8245834, 'prompt_eval_count': 2273, 'prompt_eval_duration': 96448000, 'eval_count': 280, 'eval_duration': 17390762000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0fbce-9f83-7d72-9431-9b07ead7dc29-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/note/todo1.txt'}, 'id': '0f651274-3b4a-4014-b3bf-d776ce822a35', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2273, 'output_tokens': 280, 'total_tokens': 2553}), ToolMessage(content="Error: File '/note/todo1.txt'

### FileSystemBackend (local)

In [60]:
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model=model,backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [61]:
# ====================================
# Invoke agent and ask for write file
# ====================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /note/todo1.txt with exactly this content:\n"
                    "1. Recored Video\n"
                    "2. Edit Vedio\n"
                    "3. Upload video\n"
                    "Then tell me you've done it."
                ),
            }
        ]
    }
)

print("=========================Result=========================")
print(result)
print("Content:------")
print(result["messages"][-1].content)

print(result.get("files", {}))


=========================Result=========================
{'messages': [HumanMessage(content="Create a file at /note/todo1.txt with exactly this content:\n1. Recored Video\n2. Edit Vedio\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='e21c7948-7325-46d9-adb9-65aa16da8a34'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-02T08:51:49.568319Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11707674125, 'load_duration': 5596083, 'prompt_eval_count': 2300, 'prompt_eval_duration': 404559000, 'eval_count': 186, 'eval_duration': 11272515000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0fbcf-d8ff-7991-bb5c-ac42f8b90df8-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/note/todo1.txt', 'content': '1. Recored Video\n2. Edit Vedio\n3. Upload video'}, 'id': 'a367df23-659a-40b8-929b-c07c151d43ab', 'type': 'tool_call'}], invalid_

In [62]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "note" / "todo1.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 /Users/kaushalprajapati/Personal_workspace/agent/deep_agent/note/todo1.txt
----------------------------------------
1. Recored Video
2. Edit Vedio
3. Upload video


In [63]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /note/todo1.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
The contents of /note/todo1.txt are:

1. Recored Video
2. Edit Vedio
3. Upload video

(Note: The file contains exactly 3 lines as shown.)


### Deep Agent — StoreBackend verification
Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [65]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [66]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
I've created the file `/notes/todo.txt` with your specified content:

```
1. Record video
2. Edit video
3. Upload video
```

Let me know if you need anything else!


In [67]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
The contents of /notes/todo.txt are:

1. Record video
2. Edit video
3. Upload video


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

# Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a **virtual filesystem**: its tools read and write
files using paths like `/notes/todo.txt`. But those paths are an abstraction.
The **backend** decides where that data physically lives — and that single
choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

---

## 1. StateBackend (default)

Files live **in the agent's LangGraph state** — i.e. in RAM, tied to a single
thread. They're available during the run via `result["files"]`, and you must
manually carry that state forward to keep using them.

- **Where:** in-memory dict, inside one thread's state
- **Survives:** within a single conversation/thread only
- **Gone when:** the thread ends or the process exits
- **Use it for:** ephemeral scratch space, temporary working files

## 2. FilesystemBackend

Files are written to your **real disk**, under `root_dir`. With
`virtual_mode=True`, a virtual path like `/notes/todo.txt` maps to
`root_dir/notes/todo.txt`. These are genuine files you can open in an editor
or `cat` from a terminal.

- **Where:** your actual filesystem, relative to `root_dir`
- **Survives:** across process restarts — it's a real file
- **Use it for:** when the agent should edit real project files
- **⚠️ Caution:** grants the agent real read/write disk access — point it at a
  scratch directory, not anything important

## 3. StoreBackend

Files live in a **LangGraph store**, scoped by a `namespace`. This is the only
backend whose files persist **across threads/conversations** — write on one
thread, read back on another. With `InMemoryStore` this works within a single
process; swap in a persistent store (e.g. Postgres-backed) for true
cross-restart durability.

- **Where:** inside the store object, under a namespace key
- **Survives:** across threads sharing the same store
- **Gone when:** the process exits (if using `InMemoryStore`) — use a
  persistent store to outlive restarts
- **Use it for:** long-term memory, per-user file spaces

---

### Quick comparison

| Backend            | Lives in        | Cross-thread? | Survives restart?      | Real file on disk? |
|--------------------|-----------------|:-------------:|:----------------------:|:------------------:|
| `StateBackend`     | LangGraph state |       ❌      |           ❌           |         ❌         |
| `FilesystemBackend`| Your disk       |       ✅      |           ✅           |         ✅         |
| `StoreBackend`     | A LangGraph store |     ✅      | only w/ persistent store |       ❌         |

> **Mental model:** the agent never knows the difference. Its tools always
> speak in file paths — the backend quietly translates every read/write into
> the right operation underneath. That's what lets you swap backends without
> touching a line of agent logic.